In [2]:
import json

In [1]:
json_filepath = "Assignments/self_critique_loop_dataset.json"

In [4]:
with open(json_filepath, "r",encoding = "utf-8") as f:
    json_data = json.load(f)

In [6]:
documents = []

for item in json_data:
    text = "\n".join(f"{k} : {v}" for k,v in item.items())
    documents.append(text)

In [10]:
json_data[0]

{'doc_id': 'KB001',
 'question': 'What are best practices for debugging?',
 'answer_snippet': "When addressing debugging, it's important to follow well-defined patterns...",
 'source': 'debugging_guide.md',
 'confidence_indicator': 'moderate',
 'last_updated': '2024-01-10'}

In [11]:
from langchain.docstore.document import Document
docs = [Document(page_content=text,metadata = item) for text, item in zip(documents,json_data)]

In [13]:
docs[0]

Document(metadata={'doc_id': 'KB001', 'question': 'What are best practices for debugging?', 'answer_snippet': "When addressing debugging, it's important to follow well-defined patterns...", 'source': 'debugging_guide.md', 'confidence_indicator': 'moderate', 'last_updated': '2024-01-10'}, page_content="doc_id : KB001\nquestion : What are best practices for debugging?\nanswer_snippet : When addressing debugging, it's important to follow well-defined patterns...\nsource : debugging_guide.md\nconfidence_indicator : moderate\nlast_updated : 2024-01-10")

In [14]:
from dotenv import load_dotenv

load_dotenv()

model = "text-embedding-3-large"
llm_model = "gpt-4.1-mini"
from langchain_openai import AzureOpenAIEmbeddings
embeddings = AzureOpenAIEmbeddings(model=model,api_version="2024-12-01-preview")
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(model=llm_model,api_version="2024-12-01-preview")
from langchain_chroma import Chroma

In [15]:

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)



Define a LangGraph graph containing:

    A retriever node (fetches top‐k relevant snippets),
    A single LLM‐generator node (produces an answer using retrieved context),
    A self‐critique node (checks if the answer is complete), and
    A refinement node (if critique finds gaps, retrieve one more snippet and regenerate).

Expose a minimal interface (Jupyter notebook or simple Python script) to enter a question and see the end‐to‐end pipeline run (initial answer → critique → optional refinement → final answer).


In [ ]:
from typing import TypedDict, Literal, List
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph,START,END

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

additional_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

message = """You are a software best-practices assistant.
User Question:
{question}

Retrieved Snippets:
{context}

Task:
Based on these snippets, write a concise answer to the user’s question.
Cite each snippet you use by its doc_id in square brackets (e.g., [KB004]).
Return only the answer text.
"""
ragprompt = PromptTemplate.from_template(message)

class ragState(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retriever_node(state:ragState):
    ret_docs = retriever.invoke(state["question"])
    return {"context":ret_docs}

# generation node
def generate(state:ragState):
    doc_content = "\n\n".join(doc.page_content for doc in state["context"])
    message = ragprompt.invoke({"question":state["question"],"context":doc_content})
    response = llm.invoke(message)
    return {"answer":response}

builder = StateGraph(ragState).add_sequence([retriever_node,generate])
builder.add_edge(START,"retriever_node")
builder.add_edge("generate",END)
raggraph = builder.compile()   

response = raggraph.invoke({"question":"what is large cap quity fund?"})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field


# Data Model for Binary Scoring of Document Relevance
class GradeDocuments(BaseModel):
    """A binary score to determine the relevance of the retrieved document."""

    # A field indicating whether the document is relevant to the question, represented as 'yes' or 'no'
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# An LLM that generates structured outputs using the GradeDocuments data model.
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Define system prompt
system = """You are a grader assessing relevance of a LLM genrated response to a user question. \n
    If the response contains keyword(s) or semantic meaning related to the question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

# Create chat prompt template
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

# Initialize retrieval evaluator
retrieval_grader = grade_prompt | structured_llm_grader

In [ ]:
from typing import Annotated, List
from typing_extensions import TypedDict

# Define State
class GraphState(TypedDict):
    question: Annotated[str, "The question to answer"]
    generation: Annotated[str, "The generation from the LLM"]
    documents: Annotated[List[str], "The documents retrieved"]
    refined_for_re_retrieval:bool
    relevant:bool
    answer:str

In [ ]:
from langchain.schema import Document
from langgraph.graph import StateGraph, END


# Document Retrieval Node
def retrieve(state: GraphState):
    print("\n==== RETRIEVE ====\n")
    question = state["question"]

    # Perform document retrieval.
    documents = retriever.invoke(question)
    return {"documents": documents}


# Answer Generation Node
def generate(state: GraphState):
    print("\n==== GENERATE ====\n")
    context = "\n".join([doc.page_content for doc in state["documents"]])
    message = ragprompt.invoke({"question":state["question"],"context":context})
    response = llm.invoke(message)
    print("Response: ",response)
    return {"answer": response}


# Document Evaluation Node
def grade_documents(state: GraphState):
    print("\n==== [CHECK RESPONSE RELEVANT TO QUESTION] ====\n")
    prompt = PromptTemplate.from_template("""
    Are the following documents relevant to the query?
    Query: {query}
    Documents: {docs}
    Answer "YES" or "NO".
    """)
    formatted_docs = "\n".join([doc.page_content for doc in state["documents"]])
    result = llm.invoke(prompt.format(query=state["question"], docs=formatted_docs))
    is_relevant = "yes" in result.content.lower()
    return {
        "question": state["question"],
        "documents": state["documents"],
        "relevant": is_relevant,
        "additional_retrieval": state.get("additional_retrieval")
    }

def additional_retrieve(state: GraphState):
    print("\n==== RETRIEVE ====\n")
    question = state["question"]

    # Perform document retrieval.
    documents = additional_retriever.invoke(question)
    return {"documents": documents}


# def additional_retrieval(state: GraphState):
#     print("\n==== [ADDITIONAL RETRIEVAL] ====\n")
#     question = state["question"]

#     if not state.get("refined_for_re_retrieval"): # for first time query finetuning for Vector DB
#         prompt = f"Generate 5 improved variants of the following query: {question}"
#         response = llm.invoke(prompt)
#         improved_query = question + " " + " ".join(response.content.split("\n"))
#         return {"question": improved_query, "refined_for_re_retrieval": True}
#     else:
#         # Rewrite the question.
#         better_question = question_rewriter.invoke({"question": question})
#         return {"question": better_question, "web_search": True}


In [ ]:
def build_retrieval_grader_subgraph():
    sub_builder = StateGraph(GraphState)
    sub_builder.add_node("Retrieve", retrieve)
    sub_builder.add_node("Generate", generate)
    sub_builder.add_node("Grade", grade_documents)
    sub_builder.add_edge("Generate", "Grade")
    sub_builder.add_edge("Grade", END)
    return sub_builder.compile()

retrieval_grader = build_retrieval_grader_subgraph()


# ---- Main Graph ----
builder = StateGraph(GraphState)

builder.add_node("RetrievalGrader", retrieval_grader)
builder.add_node("Generator", generate)
builder.add_node("AdditionalRetriever", additional_retrieve)

builder.set_entry_point("RetrievalGrader")

# ---- Routing Logic ----
def route_after_grading(state: GraphState) -> str:
    if state["relevant"]:
        return "Generator"
    else:
        return "AdditionalRetriever"


builder.add_conditional_edges("RetrievalGrader", route_after_grading,{"Generator":"Generator",
                                                                      "AdditionalRetriever":"AdditionalRetriever",
                                                                      
                                                                      })
builder.add_edge("AdditionalRetriever","Generator")
builder.add_edge("Generator", END)
graph = builder.compile()
